*Année universitaire 2025 - 2026*  


Nom : BAKALA / Prénom : Sergina Esther

L'étude du jeu de données _BaseCLD.csv_ nous permettra de produire des indicateurs synthétiques et de développer des outils d’aide à la décision. Ces derniers viseront à mieux caractériser les niveaux de contamination et à éventuellement identifier les zones à risque.

In [ ]:
pip install plydata

# II. Analyse préliminaire

In [ ]:
# Chargement des packages d'intérêt
import numpy as np
# Import de plydata qui pose problème import plydata as ply
import pandas as pd
import seaborn as sns
from plotnine import * 
from sklearn import cluster
from sklearn.tree import DecisionTreeClassifier
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage
from scipy.cluster import hierarchy
import scipy
import scipy.stats
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
import pylab
import scipy.stats as stats
from scipy.stats import ks_2samp
import os
import folium
import xyzservices.providers as xyz
import geopandas as gpd
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt
from folium.plugins import HeatMapWithTime
import branca
import unicodedata
import re
from pyproj import Transformer

In [ ]:
# Chargement des données
Chlordecone = pd.read_csv("BaseCLD2026.csv", sep=";", decimal=',')

# Vérifications de la taille du dataframe
Chlordecone.shape

Nous travaillons sur des observations réalisés sur près de **31126 parcelles agricoles**. Pour chacune de ces parcelles nous avons exactement **22 caractéristiques** comme l'année de prélèvement, le taux de chlordécone, le type de sol ou encore la moyenne de la rugosité du terrain.

In [ ]:
# Aperçu des premières lignes du dataframe
Chlordecone.head(2)

Le dataframe a bien été importé. Cette première visualisation permet d'identifier plusieurs types de données (quantitatives et qualitatives ) qu'il faudra contrôler. Leurs types, et l'information auxquelles elles renvoient sont des éléments nous guiderons dans les contrôles à effectuer.

On remarque notamment que les variables contenant des valeurs décimales comme _Moyenne de l’indice topographique_  sont bien formatés contrairement à  la variable _Taux_5b_hydro_ malgré le paramètre decimal lors de la lecture du fichier csv. Pandas ne pourra donc pas l'interpréter comme une valeur numérique. Nous allons donc devoir remplacer manuellement la virgule par le point.

In [ ]:
# Remplacement de la virgule par le point
Chlordecone['Taux_5b_hydro'] = Chlordecone['Taux_5b_hydro'].str.replace(',', '.')

## II-1. Vérification des doublons

In [ ]:
# Identification des doublons parfaits
doublons_stricts = Chlordecone.duplicated().sum()
print(doublons_stricts)

Il semble y avoir 8 lignes en trop c'est-à-dire qu'elles ont les mêmes valeurs pour les différentes variables du dataframe que d'autres lignes.

In [ ]:
# Affichage des premières lignes répétées pour analyse
# Chlordecone[Chlordecone.duplicated(keep=False)]

On aurait pu supposer que ces données prenaient en compte des paramètres susceptibles de fluctuer au cours de la journée (par exemple, plusieurs prélèvements effectués le même jour). Cependant, la fonction duplicated, qui considère l’ensemble des variables pour la gestion des doublons, ainsi que l’examen visuel des données, montrent que les lignes dupliquées sont de simples copies d’observations existantes. Elles n’apportent donc aucune information supplémentaire à l’analyse et ne font qu’alourdir le dataframe. Nous allons par conséquent les supprimer.

In [ ]:
# Suppression des lignes en doublons
Chlordecone_ = Chlordecone[~Chlordecone.duplicated()]

# Vérification de la taille du dataframe après suppression
Chlordecone_.shape

Le nombre de lignes a été réduit de 8, la suppression a donc été un succès.

## II-2. Vérification du type des variables

In [ ]:
Chlordecone_.dtypes

Pour la suite du travail, nous allons convertir quelques colonnes dans le bon type afin qu'elles soient correctement interprétées. 

In [ ]:
# Conversion des colonnes en types date
Chlordecone_['Date_prelevement'] = pd.to_datetime(Chlordecone_['Date_prelevement'], dayfirst=True, errors='coerce')
Chlordecone_['Date_enregistrement'] = pd.to_datetime(Chlordecone_['Date_enregistrement'], dayfirst=True, errors='coerce')
Chlordecone_['Date_analyse'] = pd.to_datetime(Chlordecone_['Date_analyse'], dayfirst=True, errors='coerce')

# Conversion des colonnes adéquates en float
float_cols = [
    'Taux_5b_hydro', 'Taux_Chlordecone', 'mnt_tpi_mean', 'mnt_tri_mean', 
    'mnt_rugosite_mean', 'mnt_ombrage_mean', 'mnt_exposition_mean', 'mnt_pente_mean', 'X'
,'Y']
for col in float_cols:
    Chlordecone_[col] = pd.to_numeric(Chlordecone_[col], errors='coerce')

# Conversion des colonnes adéquates en string
Chlordecone_['RAIN'] = Chlordecone_['RAIN'].astype(str)
Chlordecone_['COMMU_LAB'] = Chlordecone_['COMMU_LAB'].astype(str)
Chlordecone_['type_sol'] = Chlordecone_['type_sol'].astype(str)

# Vérification des nouveaux types des colonnes
#Chlordecone_.dtypes

Les colonnes liées aux opérateurs ont été difficiles à comprendre mais il s'avère qu'elles sont cruciales dans l'étude des données.

Source : https://www.statistiques.developpement-durable.gouv.fr/sites/default/files/2023-06/note_methode_estimer_une_moyenne_sur_donnees_censurees.pdf?


In [ ]:
# Le dictionnaire de correspondance
mapping_interpretation = {
    '<': 'Limite de détection (Censure à gauche)',
    '=': 'Valeur exacte',
    '>': 'Saturation (Censure à droite)',
    'inf':'Limite de détection (Censure à gauche)'
}

# Création de la nouvelle colonne
Chlordecone_['Interpretation_5b'] = Chlordecone_['Operateur_5b'].map(mapping_interpretation)

# Création de la nouvelle colonne
Chlordecone_['Interpretation_chlordecone'] = Chlordecone_['Operateur_chld'].map(mapping_interpretation)

La nouvelle colonne _Interpretation_5b_ simplifiera la lecture de la variable _Operateur_5b_.

# III. Analyse exploratoire

## III-1. Valeurs uniques

In [ ]:
# Calcul du nombre de valeurs uniques et du ratio
df_unique = Chlordecone_.nunique().to_frame(name='Valeurs Uniques')
df_unique['% de diversité'] = (df_unique['Valeurs Uniques'] / len(Chlordecone_) * 100).round(2)
df_unique = df_unique.sort_values(by='Valeurs Uniques', ascending=False)

# Affichage avec style (couleurs)
df_unique.style.background_gradient(cmap='Blues', subset=['Valeurs Uniques'])

 L'étude porte sur 3 619 parcelles agricoles différentes dans 36 communes suivies durant une décennie (10 ans). Ce qui est plutôt surprenant car la plupart des communes sont de la Martinique qui n'en compte que 34 et non 36 (un point d'attention sera porté à cette variable dans la suite du travail).L'écart entre le nombre de parcelles et le volume total du jeu de données s'explique logiquement par la répétition des prélèvements : soit par plusieurs prélèvements au cours des années dans la même parcelle, soit par des prélèvements à différents points au sein d'une même parcelle (à la même date ou pas), d'où la présence des coordonnées géographiques différents et en nombre important. Les étapes suivantes permettront de clarifier ces hypothèses.

## III-2. Distributions variables quantitatives / qualitatives

In [ ]:
var_quant=["Taux_Chlordecone","Taux_5b_hydro","histoBanane_Histo_ban","mnt_tpi_mean", "mnt_tri_mean","mnt_rugosite_mean", "mnt_ombrage_mean", "mnt_exposition_mean", "mnt_pente_mean"]
print( Chlordecone_[var_quant].describe().T)

Le résumé statistique des variables quantitatives présentent des informations tels que la moyenne, le nombre de valeur ou encore le minimum et le maximum. Une rapide appréciation des résultat montre de forte disparité au niveau des moyennes des paramètres ombrage, exposition, pente et rugosité. Un élément néanmoins surprenant est le fait que la variable _Taux_5b_hydro_ normalement quantitative semble contenir des chaines de caractère "inf" (qu'il faudra donc corriger).

In [ ]:
# On choisit les variables à afficher 
cols_to_plot = ['Interpretation_5b', 'COMMU_LAB', 'ANNEE']

fig, axes = plt.subplots(len(cols_to_plot), 1, figsize=(8, 13))

for i, col in enumerate(cols_to_plot):
    sns.countplot(data=Chlordecone_, y=col, ax=axes[i], palette='viridis', order=Chlordecone_[col].value_counts().index)
    axes[i].set_title(f'Répartition par {col}')
    axes[i].set_xlabel('Nombre de prélèvements')

plt.tight_layout()
plt.show()

On remarque déjà que la plupart des prélèvements environ 1/10e ont été effectué dans la commune de Morne-rouge ce qui peut entraîner un biais (asymétrie dans la représentativité). Cependant des recherches sur le net peuvent justifier la sur-représentation de cette commune par son histoire agricole. Mais surtout le plus choquant est qu'il y a des données hors Martinique qui peuvent ne pas faire partie du périmètre d'étude si elles proviennent de la métropole. Aussi, on observe une catégorie de nan, donc de prélèvements pour lesquels les communes n'ont pas été noté. Enfin plus de 8000 prélèvements ont été effectué en 2018 mais très de peux de valeurs qu'on peut considérer exact dans l'estimation du taux de chlordecone (une part non négligeable de censure à gauche).

In [ ]:
cols = ['type_sol', 'Sol_simple']
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for i, col in enumerate(cols):
    data = Chlordecone_[col].value_counts().head(10)
    axes[i].pie(data, labels=data.index, autopct='%1.1f%%', 
                colors=sns.color_palette('magma', 10), startangle=140)
    axes[i].set_title(f'Top 10 : {col}', fontweight='bold')

plt.tight_layout()
plt.show()

De même que la variable sol simple, on constate la présence de "nan" comme une catégorie à part entière de type de sol alors qu'elle doit normalement correspondre à des valeurs vides et qui seront par conséquent prédites.
On remarque surtout qu'il y a un type de sol_simple qui s'appelle No data, ce qui peut poser problème dans la suite de notre travail car "No data" risque d'être considéré comme une catégorie 

# III-4. Vérification des valeurs nulles / manquantes

In [ ]:
# Calcul du nombre total pour l'affichage
total_nan = Chlordecone_.isnull().sum()
print("Nombre de valeurs manquantes par colonne :")
print(total_nan[total_nan > 0]) # Affiche seulement celles qui ont des manques

# Visualisation des valeurs manquantes
sns.heatmap(Chlordecone_.isnull(), cbar=True, cmap="YlGnBu")
plt.title("Répartition des valeurs manquantes")
plt.show()

## III-5. Cohérence métier (dates)

In [ ]:
df = Chlordecone_.query("~(Date_enregistrement <= Date_analyse)")
len(df)

La logique souhaiterait qu'une fois l'échantillon arrivé au laboratoire qu'il soit d'abord enregistré puis analyser. Le fait que 3886 échantillons aient d'abord été analysé avant d'être enregistré reflète soit d'une erreur de saisie ou un problème de traçabilité de la part du laboratoire.

# IV. Data preparation

## IV-1. Suppression données hors contexte

L'objectif de notre travail et d'étudier le taux de chlordecone aux antilles. Toutes les communes étant issues de Martinique, la commune intitulé "Hors - Martinique " est l'une des raison qui fait porté le nombre de communes distinctes à 36. Cete catégorie pollue le jeux de données car elle est hors du contexte géographique. C'est la raison pour laquelle nous allons la supprimer du dataframe.

In [ ]:
# Crétation d'un nouveau dataframe 
Chlordecone_MTQ = Chlordecone_[Chlordecone_['COMMU_LAB'] != 'Hors Martinique']
Chlordecone_MTQ.shape

## IV-2. Gestion des communes vides 

In [ ]:
# Nombre de lignes pour lesquels la communes n'est pa mentionnée.
len(Chlordecone_MTQ[Chlordecone_MTQ['COMMU_LAB']=='nan'])

Comme nous avons déjà les noms des communes, le modèle k-NN est bien adapté pour retrouver les communes manquantes à partir des coordonnées géographiques.

In [ ]:
# 1. Masque et séparation
mask_vide = Chlordecone_MTQ['COMMU_LAB']=="nan"
train_data = Chlordecone_MTQ[~mask_vide]
predict_data = Chlordecone_MTQ[mask_vide]

# 2. KNN 
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(train_data[['X', 'Y']], train_data['COMMU_LAB'])

# 3. Prédiction et injection
predictions = knn.predict(predict_data[['X', 'Y']])
Chlordecone_MTQ.loc[mask_vide, 'COMMU_LAB'] = predictions

print(f"Nettoyage terminé : {len(predictions)} communes identifiées grâce aux coordonnées.")

## IV-3. Imputation variable sol simple

Nous avons vu plus haut que la colonne « Sol simple » contient une catégorie no data, qui correspond en réalité à un manque d’information. Nous allons donc la remplacer par none.

In [ ]:
Chlordecone_MTQ = Chlordecone_MTQ.replace("No data", None)

Pour prédire les types de sols manquants, nous avons privilégié l'utilisation des coordonnées géographiques X et Y et les communes. En effet :
- les variables de types dates présentent parfois une incohérence chronologiques entre les différentes étapes (prélèvement, enregistrement et analyse).
  
- La plupart des variables quantitatives (les propriétés chimiques des sols ou le taux de chlordecone ) possèdent des valeurs manquanes ce qui rend leurs utilisations inadaptés pour la prédiction.  

- Les coordonnées géographiques d'un point de vue logique semblent le mieux adaptés car deux sols géographiquement proches ont de fortes probabilités d'être du même type. On aurait pu rajouter les communes dans l'algorithme de prédiction pour le rendre plus robuste mais ce dernier utilisant mieux des variables quantitatives, il faut l'encoder.

In [ ]:
# 1. Préparation des colonnes (X, Y et Commune encodée)
le = LabelEncoder()
Chlordecone_MTQ['commune_num'] = le.fit_transform(Chlordecone_MTQ['COMMU_LAB'])

features = ['X', 'Y', 'commune_num']
mask_missing = Chlordecone_MTQ['Sol_simple'].isna()

# 2. Normalisation et séparation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(Chlordecone_MTQ[features])

X_train = X_scaled[~mask_missing]
y_train = Chlordecone_MTQ.loc[~mask_missing, 'Sol_simple']
X_test = X_scaled[mask_missing]

# 3. Entraînement et Remplissage
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, y_train)

Chlordecone_MTQ.loc[mask_missing, 'Sol_simple'] = knn.predict(X_test)

print(f" {mask_missing.sum()} sols complétés avec succès.")

## IV-4. Imputation variable type sol

La même approche a été gardé pour la variable type sol.

In [ ]:
# 1. Préparation des colonnes (X, Y, Commune encodée et sol simple)
le_commune = LabelEncoder()
le_sol = LabelEncoder()

Chlordecone_MTQ['commune_num'] = le_commune.fit_transform(Chlordecone_MTQ['COMMU_LAB'])
Chlordecone_MTQ['sol_num'] = le_sol.fit_transform(Chlordecone_MTQ['Sol_simple'])

features = ['X', 'Y', 'commune_num', 'sol_num']
mask_missing = Chlordecone_MTQ['type_sol']=="nan"

# 2. Normalisation et séparation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(Chlordecone_MTQ[features])

X_train = X_scaled[~mask_missing]
y_train = Chlordecone_MTQ.loc[~mask_missing, 'type_sol']
X_test = X_scaled[mask_missing]

# 3. Entraînement et Remplissage
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(X_train, y_train)

Chlordecone_MTQ.loc[mask_missing, 'type_sol'] = knn.predict(X_test)

print(f" {mask_missing.sum()} types sols complétés avec succès.")

## IV-5. Gestion variables commençant par mnt

On constate que les variables correspondant aux moyennes des paramètres météorologiques ou climatiques présentent le même nombre de valeurs nulles. Cela suggère l’existence d’un lien entre elles, notamment en raison de prélèvements qui n’ont peut-être pas pu être réalisés dans certaines zones.

La première étape est de vérifier si les lignes vides sont les mêmes pour toutes les variables.

In [ ]:
# On récupère les indices des lignes nulles pour deux variables
null_tpi = Chlordecone_MTQ[Chlordecone_MTQ['mnt_tpi_mean'].isnull()].index
null_pente = Chlordecone_MTQ[Chlordecone_MTQ['mnt_pente_mean'].isnull()].index

# On compare : si True, les 28 lignes sont exactement les mêmes
print(null_tpi.equals(null_pente))

Nous avons vu que toutes ces variables sont nulles pour les même lignes, il s'agit de vérifier s'il s'agit d'une même zone géographique ( commune en particulier) ou d'un type de sol précis.

In [ ]:
Chlordecone_MTQ[Chlordecone_MTQ['mnt_pente_mean'].isna()]['COMMU_LAB'].value_counts()

In [ ]:
Chlordecone_MTQ[Chlordecone_MTQ['mnt_pente_mean'].isna()]['Sol_simple'].value_counts()

Les valeurs manquantes ne concernent pas un seul type de sol.
Ces deux études nous permet de constater qu'il ne s'agit pas de NMAR, Non missing at random. On est plus de une situation de MAR (missing at random).

Pour éviter d’attribuer des valeurs aléatoires qui pourraient fausser les indicateurs, on peut remplacer les valeurs manquantes dans les colonnes concernées par la moyenne des observations disponibles. Ce choix se justifie par la très faible proportion de données manquantes par rapport au nombre total de lignes (0,1 % des données).

In [ ]:
# 1. Liste des variables à observer
cols_to_plot = ['mnt_pente_mean', 'mnt_exposition_mean', 'mnt_ombrage_mean', 'mnt_rugosite_mean', 'mnt_tri_mean', 'mnt_tpi_mean']

# 2. Préparation des données (Format "Long")
# On transforme les colonnes en lignes pour que Seaborn puisse les comparer
df_melted = Chlordecone_MTQ.melt(value_vars=cols_to_plot)

# 3. Création du graphique
plt.figure(figsize=(10, 6))
sns.boxplot(x='variable', y='value', data=df_melted, palette="Set3")

plt.title('Comparaison des distributions et détection des outliers')
plt.xlabel('Variables Météo')
plt.ylabel('Valeurs')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Le boxplot des variables des conditions météorologiques présentent un nombre important de données aberrantes. Ces dernières rendent risquée l'imputation par la moyenne. Il faut commencer par les traiter. Il est assez délicats de les supprimer car la richesse de climat et de végétations de la martinique peuvent expliquer ces valeurs "aberrantes" qui ne le sont pas forcéments. Aussi, ces variables correspondent elles mêmes à des moyenne pour chaque parcelle, aussi une approche non réfléchie sur leur traitement peut avoir une incidence sur les observations qui ont servit à les agréger.

In [ ]:
# les variables de moyennes par parcelle
cols = ['mnt_pente_mean', 'mnt_rugosite_mean', 'mnt_tri_mean', 'mnt_tpi_mean', 'mnt_ombrage_mean', 'mnt_exposition_mean']

# Calcul de la moyenne, médiane et du coefficient d'asymétrie
stats_comp = pd.DataFrame({
    'Moyenne': Chlordecone_MTQ[cols].mean(),
    'Médiane': Chlordecone_MTQ[cols].median(),
    'skewness ': abs(3*(Chlordecone_MTQ[cols].mean() - Chlordecone_MTQ[cols].median()) / Chlordecone_MTQ[cols].std())
})

print(stats_comp.round(2))

Le coefficient d'asymétrie avec la  méthode skewness est un bon moyen d'avoir un aperçu sur l'asymétrie de chacune des ces variables et donc du poids que les valeurs extrêmes ont sur l'ensemble de la distribution.
Pour les variables on constate un coefficient compris entre 0.5 et -0.5, ce qui signifie que ces variables ne présentent pas une grande asymétrie, une asymétrie très négligeables et tendent plus vers la symétrie (https://www.datacamp.com/fr/tutorial/understanding-skewness-and-kurtosis).

Nous n'allons donc pas les transformer si ce n'est que les standardiser afin qu'elles aient le même poids pour la suite du travail.

In [ ]:
# Imputation des valeurs manquantes par la moyenne
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(Chlordecone_MTQ[cols])

# Standardisation robuste 
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_imputed)

# Remplacer les colonnes originales par les données transformées
Chlordecone_MTQ[cols] = X_scaled

## IV-6. Gestion de la variable Taux 5b hydro

La variable taux 5b hydro, qui est un métabolite (dérivé) de la chlordecone présente des valeurs atypiques de type chaine de caractère : 'inf' que nous devons corriger.

In [ ]:
np.isinf(Chlordecone_MTQ["Taux_5b_hydro"]).sum()

Il y a 586 valeurs inf qui correspondent peut être à des erreurs de saisie. Avant de procéder à leurs traitements, nous allonrs étudier les valeurs aberrantes.

In [ ]:
Chlordecone_MTQ["Taux_5b_hydro"].plot(kind='box')
plt.title("Distribution Taux 5b_hydro")
plt.show()

Le boxplot présente des valeurs aberrantes apparantes. Nous allons procéder à une winsorisation c'est à dire ramener les valeurs extrêmes maximale à la valeur Q3 + 1,5 * IQR. (source : https://www.calculatorultra.com/fr/tool/quartile-and-interquartile-range-iqr-calculator.html)

In [ ]:
# Remplacement des chaines inf par nan
Chlordecone_MTQ["Taux_5b_hydro"] = Chlordecone_MTQ["Taux_5b_hydro"].replace([np.inf, -np.inf], np.nan)

#  Calcul des seuils (sur les données réelles correctes hors nulles)
valeurs_reelles = Chlordecone_MTQ.loc[Chlordecone_MTQ["Taux_5b_hydro"].notna(),"Taux_5b_hydro"]

Q1, Q3 = valeurs_reelles.quantile([0.25, 0.75])
IQR = Q3 - Q1

seuil_max = Q3 + 1.5 * IQR
seuil_mini = Q1 - 1.5 * IQR

# Sécurité : Le seuil mini ne peut pas être inférieur à 0 pour une concentration
if seuil_mini < 0:
    seuil_mini = 0 

print(f"Bornes de l'étude : [{seuil_mini:.4f} ; {seuil_max:.4f}]")

# Traitement des valeurs Hors-Limites
lim_haut = (Chlordecone_MTQ["Taux_5b_hydro"] > seuil_max)
Chlordecone_MTQ.loc[lim_haut, "Taux_5b_hydro"] = seuil_max

# Plafonnement vers le bas (Valeurs trop basses)
lim_bas = (Chlordecone_MTQ["Taux_5b_hydro"] < seuil_mini)
Chlordecone_MTQ.loc[lim_bas, "Taux_5b_hydro"] = seuil_mini

Nous allons dans la suite transformer la valeur avec des 'inf' en Nan et les imputer en utilisant surtout les coordonnées géographiques et l'algorithme KNN. La particularité est que la fonction knn utilisé sera de type regression au vue de la nature de la variable à prédire : quantitative.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score

# 1Détecter les valeurs manquantes
mask = Chlordecone_MTQ['Taux_5b_hydro'].isna()

# 2Encoder les variables qualitatives
le_commune = LabelEncoder()
le_sol = LabelEncoder()
le_type = LabelEncoder()
Chlordecone_MTQ['commune_encode'] = le_commune.fit_transform(Chlordecone_MTQ['COMMU_LAB'])
Chlordecone_MTQ['sol_encode'] = le_sol.fit_transform(Chlordecone_MTQ['Sol_simple'])
Chlordecone_MTQ['type_sol_encode'] = le_type.fit_transform(Chlordecone_MTQ['type_sol'])

# Préparer les variables explicatives
features = ['X', 'Y', 'commune_encode', 'sol_encode', 'type_sol_encode']
X = Chlordecone_MTQ[features]

# Normaliser les données pour KNN
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Séparer données connues et à compléter
X_known = X_scaled[~mask]
y_known = Chlordecone_MTQ.loc[~mask, 'Taux_5b_hydro']
X_to_fill = X_scaled[mask]

# Créer et évaluer le modèle
knn = KNeighborsRegressor(n_neighbors=10, weights='distance')
score = cross_val_score(knn, X_known, y_known, cv=5, scoring='r2')
print(f"Score R² moyen : {score.mean():.2f}")

# Remplir les valeurs manquantes
knn.fit(X_known, y_known)
Chlordecone_MTQ.loc[mask, 'Taux_5b_hydro'] = knn.predict(X_to_fill)

print(f"{mask.sum()} valeurs complétées avec succès.")

On obtient un score R2 pas excellent mais moyen pour coninuer avec l'analyse.

## IV-7. Gestion de la variable cible

Lorsque nous parlons de variable cible, nous faisons référence à la variable taux de chlordecone. Nous avons allons essayer de visualiser la distribution des données sachant que nous avons déjà obtenu plus haut des statistiques élémentaires sur cette variable.

In [ ]:
(
    ggplot(Chlordecone_MTQ, aes(x="Taux_Chlordecone")) 
    + geom_histogram(bins=30, color="white")
    + theme_minimal()
    + labs(
        title="Distribution des taux de Chlordécone",
        x="Concentration",
        y="Fréquence"
    )
)

On constate une forte proportion de taux de chlordecone presque nulle (sûrement la classe modale). Pourtant on constate qu'il y a quelques valeurs concentrations assez éloignés des autres typiquement de plus de 5 et environ 20 qui se distinguent assez. Cela peut laisser penser à des valeurs atypiques que nous confirmerons dans la suite du travail. Pour l'instant nous nous rendons bien compte que l'histogramme n'est pas suffisant pour identifier la distribution ou la loi que suit la variable.

In [ ]:
stats.probplot(Chlordecone_MTQ["Taux_Chlordecone"], dist="norm", plot=pylab)

plt.xlabel('Quantiles théoriques')
plt.ylabel('Quantiles observés')
plt.title('qq-plot - Droite de Henry')
plt.grid(True, linestyle = '--')

Nous avons testé la distribution normale des données, grâce à un QQ-plot avec la droite de Henry. On remarque assez rapidement que la droite et les points ne sont pas alignés. Ce qui suggère qu'on est très loin d'une distribution normale.

In [ ]:
x =  Chlordecone_MTQ["Taux_Chlordecone"]
shapiro_test = stats.shapiro(x)
shapiro_test

Le test de shapiro-wilk fournit une pvalue de 0, on peut donc rejeter l'hypothèse initiale de normalité. Pour la partie data analyse, il faudrait donc normaliser les données.

In [ ]:
# Source : https://www.stat4decision.com/fr/distribution-donnees-python/

data = Chlordecone_MTQ["Taux_Chlordecone"]

### Histogramme des données
y, x = np.histogram(data, bins=20, density=True)
# Milieu de chaque classe
x = (x + np.roll(x, -1))[:-1] / 2.0

# Liste des lois à tester
dist_names = ['norm', 'gamma', 'pareto', 'lognorm', 'invgamma', 'invgauss', 'loggamma', 'alpha', 'chi', 'chi2']

sse = np.inf
sse_thr = 0.05

best_name = ""
best_pdf = None

# Pour chaque distribution
for name in dist_names:

	# Modéliser
	dist = getattr(scipy.stats, name)
	param = dist.fit(data)

	# Paramètres
	loc = param[-2]
	scale = param[-1]
	arg = param[:-2]

	# PDF
	pdf = dist.pdf(x, *arg, loc=loc, scale=scale)
	# SSE
	model_sse = np.sum((y - pdf)**2)

	# Si le SSE est ddiminué, enregistrer la loi
	if model_sse < sse :
		best_pdf = pdf
		sse = model_sse
		best_loc = loc
		best_scale = scale
		best_arg = arg
		best_name = name

	# Si en dessous du seuil, quitter la boucle
	if model_sse < sse_thr :
		break

# 3. Affichage
plt.figure(figsize=(10, 6))
plt.hist(data, bins=30, density=True, alpha=0.3, label="Données (Histogramme)", color="gray")
plt.plot(x, best_pdf, label=f"Meilleur fit : {best_name}", linewidth=3, color="red")
plt.legend()
plt.title(f"Identification de la loi : {best_name}")
plt.show()

print(f"Modèle sélectionné : {best_name}")
print(f"Erreur (SSE) : {sse}")

La loi de Pareto est celle qui minimise la somme des carrées des erreurs pour laquelle on obtient une valeur inférieur au seuil de 0.05. Elle semble donc être celle qui décrit le mieux la distribution de la variable chhlordecone. 
La loi de pareto est encore appelé la loi des 80/20, cela signifie que 80% des taux de chlordecone sraient concentrés dans 20% des parcelles agricoles de Martinique.

Pour l'identification des valeurs aberrantes en partie mise en évidence avec l'histogramme, et sans contrainte métier, nous avons fixé un seuil arbitraire de 10. C'est à dire qu'au delà de cette valeur, un taux sera considéré comme aberrant. Pour limiter l'influence de ces valeurs dans la suite du travail, nous les imputerons par la médiane.

In [ ]:
ggplot(Chlordecone_MTQ, aes(x=0,y="Taux_Chlordecone")) + geom_boxplot() + theme_minimal()

In [ ]:
# Calcul de la mediane
med = Chlordecone_MTQ["Taux_Chlordecone"].median()

aberrantes = (Chlordecone_MTQ["Taux_Chlordecone"]>10) 

# Nombre de valeurs pour lesquelles le taux de chlordecone est supérieur à 10
(Chlordecone_MTQ["Taux_Chlordecone"]>10).sum() 

In [ ]:
# Imputation par la médiane
Chlordecone_MTQ.loc[Chlordecone_MTQ["Taux_Chlordecone"] > 10, "Taux_Chlordecone"] = med

# Vérification de la nouvelle distribution
stat, p_value = ks_2samp(x, Chlordecone_MTQ["Taux_Chlordecone"])

print(f"Statistique de KS : {stat:.4f}")
print(f"p-value : {p_value:.4f}")

Une fois l'imputation par la médiane réalisée, on réeffectue le test de kolmogorov et on constate que la P value est toujours nulle. Sachant que notre hypothèse initiale était que les deux distributions étaient similaires, on peut la rejeter car la valeur étant inférieure à 0.05, cela signifie quela distribution a grandement changé à cause du pic crée par la médiane.

## IV-8. Gestion des autres variables

La variable « histoBanane_Histo_ban » ne possède que deux modalités et présente un grand nombre de valeurs nulles : plus de 50 % des lignes sont vides.  Mais elle permet de mettre en évidence la présence de bananeraies dans les années antérieurs ( 1960, 1970, 1980). Nous allons créer une colonne histoBanane_bool qui vaut 1 si le terrain a un historique et 0 sinon donc nan.

In [ ]:
Chlordecone_MTQ['histoBanane_bool'] = Chlordecone_MTQ["histoBanane_Histo_ban"].notna().astype(int) 

De plus au vu des incohérences métiers entre les dates d'analyse, et de prélèvement au niveau du laboratoire et tenant compte du nombre important de valeurs manquantes dans les dates d'analyse, nous ne garderons dans notre dataframe final que les dates de prélèvements qui appore plus d'information sur le sol à l'instant T du prélèvement. Elle est plus pertinente pour suivre l'évolution du taux de pollution au Chlordecone.

De même les opérateurs ont été recodée afin de faciliter leurs lectures et inteprétation. Ils n'ont donc plus aucun intérêt dans le dataframe. De même la variable id permet uniquement d'identifier les différentes parcelles agricoles, or on sait qu'une ligne correspond à une parcelle donc en soit elle n'apporte rien.

In [ ]:
Chlordecone_MTQ.drop(["Date_enregistrement", "Date_analyse", 'commune_num', 'sol_num', 'Operateur_5b', 'Operateur_chld', 'ID'] , axis=1, inplace=True)

## IV-9. Extraction des composant de dates

Nous allons extraire pour la seule date conservée donc la date de prélèvement, les composants en prêtant une attention particulière aux mois car les saisons pourraient influencer le taux de chlordecone et aux années, avec peut - être une diminution ou une évolution du taux depuis l'uilisation intensive de ce produit dans les années 1900. 
Aussi, la variable *Annee* qui correspond soit à l'année d'observation soit à l'année de prélèvement sera remplacée par l'année extraite de la date de prélèvement. Cela garantie qu'on a la même observation/information pour toutes les parcelles.

In [ ]:
Chlordecone_MTQ["ANNEE_PRELEVEMENT"] = Chlordecone_MTQ["Date_prelevement"].dt.year
Chlordecone_MTQ["MOIS"] = Chlordecone_MTQ["Date_prelevement"].dt.month
Chlordecone_MTQ["JOUR"] = Chlordecone_MTQ["Date_prelevement"].dt.day

In [ ]:
Chlordecone_MTQ = Chlordecone_MTQ.drop("Date_prelevement", axis=1)
Chlordecone_MTQ.head(3)

## V. Data analysis

Dans cette partie du travail nous allons tenter de donner un sens à l'ensemble de dataset sur la Chlordecone. Il s'agira donc de mettre en évidence les relations existantes entre les différentes variables, d’étudier l’évolution des concentrations de Chlordecone au fil du temps, d’identifier les facteurs environnementaux susceptibles d’influencer ces concentrations, ainsi que de déterminer les types de sols les plus exposés à cette pollution (définir un profil de sols plus risqués).

Pour ce  faire nous  avons essentiellement opté pour différentes méthodes d'analyse, et différents tests pour confirmer les résultats.

### V-1.ACP

L'analyse en composante principale nous permettra de voir quelles sont les variables les plus porteuses d'informations (en lien avec la chlordecone). Cela reviendra à réduire la dimensionnalité tout en conservant un maximum de variance qui représente la richesse de notre jeu de données.

In [ ]:
quanti_vars = ['Taux_Chlordecone', 'Taux_5b_hydro', 'mnt_pente_mean', 'mnt_rugosite_mean', 'mnt_tri_mean', 'mnt_tpi_mean', 'mnt_ombrage_mean', 'mnt_exposition_mean']
quali_vars = ['Mois', 'type_sol','Jour', "ANNEE", "COMMU_LAB", "RAIN", "Sol_simple"]

from factor_analyzer.factor_analyzer import calculate_kmo
kmo_all, kmo_model = calculate_kmo(Chlordecone_MTQ[quanti_vars])
kmo_model 

On obtient un KMO d'environ à 0.7, cela signifie qu'il y a un des corrélations entre certaines variables et qu'elles se prêtent à une analyse factorielle.

In [ ]:
# instanciation 
sc = StandardScaler() 
 
#transformation – centrage-réduction 
Z = sc.fit_transform(Chlordecone_MTQ[quanti_vars]) 
print(Z)

In [ ]:
# Vérification
np.mean(Z),np.std(Z)

On obient un écart-type de 1. Le centrage et la réduction des données ont bien fonctionné.

In [ ]:
# instanciation 
acp = PCA(svd_solver='full') 

# calculs 
coord = acp.fit_transform(Z) 

#nombre de composantes calculées 
print(acp.n_components_) 

Le nombre de composants correspond bien aux nombres de variables quantitatives.

In [ ]:
n = len(quanti_vars)

In [ ]:
#valeur corrigée
valPropre = (n-1)/n*acp.explained_variance_
print(valPropre)

# Dynamique (8 ici)
plt.plot(np.arange(1, n+1), valPropre, marker='o')
plt.title("Scree plot")
plt.ylabel("Valeurs propres")
plt.xlabel("Numéro de la composante")
plt.grid(True)
plt.show()

Ce graphique nous permet de visualiser le pourcentage de variance expliquée par chaque composantes. Typiquement, par lecture graphique, on peut voir la composante 1 explique à elle seule environ 37.5% de la richesse contenue dans nos données.

In [ ]:
# 4. Cumul de variance (Très utile pour décider combien de composantes garder)
plt.plot(np.arange(1, n+1), np.cumsum(acp.explained_variance_ratio_), marker='o', color='red')
plt.ylabel("Variance cumulée")
plt.xlabel("Nombre de composantes")
plt.show()

Au vu du graphique on peut soit nous limiter à 4 ou 5 composates pour lesquelles on a près de 90% de variance expliquée (on observe une sorte de coude). L'ajout d'une 6ème ou 7 ème composante permet certe d'augmenter le pourcentage d'inertie mais de manière assez négligeable.

In [ ]:
# racine carrée des valeurs propres 
sqrt_eigval = np.sqrt(valPropre) 

#corrélation des variables avec les axes 
corvar = np.zeros((8,8)) 
 
for k in range(8): 
    corvar[:,k] = acp.components_[k,:] * sqrt_eigval[k] 

# on affiche pour les 5 premiers axes 
print(pd.DataFrame({'id':quanti_vars,'COR_1':corvar[:,0],'COR_2':corvar[:,1], 'COR_3':corvar[:,2], 'COR_4':corvar[:,3], 'COR_5':corvar[:,4]})) 

On remarque que le premier axe explique plus des variable lié aux caractéristiques du relief, tandis quee l'axe 2 est celui plus riche en information en expliquant grandement la chlordecone, et le taux de 5b hydro donc sur un aspect un peu plus chimique en terme de concentration de substance. Bien que le graphique de pourcentage de variance expliquée recommandait d'utiliser 5 axes, cela reste cependant difficile à modéliser et nous nous appuierons dans la suite du travail sur les deux premières composantes (qui sont les plus riches).

In [ ]:
# figure
plt.figure(figsize=(5,5))
plt.xlim(-1.1, 1.1)
plt.ylim(-1.1, 1.1)

#  axes horizontaux et verticaux
plt.axhline(0, color='grey')
plt.axvline(0, color='grey')

# cercle unité
cercle = plt.Circle((0, 0), 1, fill=False)
plt.gca().add_artist(cercle)

# flèches des variables
for i in range(len(quanti_vars)):
    x = corvar[i, 0]
    y = corvar[i, 1]
    
    plt.arrow(0, 0, x, y, head_width=0.03)
    plt.text(x*1.1, y*1.1, quanti_vars[i])

# titres
plt.title("Cercle des corrélations (F1 et F2)")
plt.xlabel("F1")
plt.ylabel("F2")

plt.show()

Le cercle de corrélation confirme l'analyse que vous avons faite plus haut. Nous allons donc maintenant essayer de classer nos individus ou communes. Cependant on peut aussi remarque que les éléments chimiques et les sols forment presquent un angle de 90 degré. Cela veut dire qu'il y a une indépendance locale ou une absence de corrélation linéaire directe entre ces deux variables sur ce plan factoriel précis. Cela suggère que le type de sol n'explique pas à lui seul la variation quantitative de la chlordécone. L'exposition quant à elle semble expliquer un peu plus la chlordecone.

In [ ]:
# On crée le graphique pour les deux premiers axes
plt.figure(figsize=(5, 5))
plt.scatter(coord[:, 0], coord[:, 1], alpha=0.5)
plt.xlabel('Relief')
plt.ylabel('Chimie Chlordecone')
plt.show()

On constate une forte concentration des points, on peut dire que la majortié des sols avec une faible rugosité, pente ont des taux de chlordecone relativement bas. Mais ce nuage de points reste encore assez illisible et ne permet pas assez de distinguer les individus. C'est pour cela que nous allons rajouter les variables qualitatives en tant que variables explicatives à notre ACP.

In [ ]:
# DataFrame pour visualiser l'ACP
df_pca_sol = pd.DataFrame({
    'F1': coord[:, 0],
    'F2': coord[:, 1],
    'Sol': Chlordecone_MTQ["Sol_simple"]
})

#  nuage de points coloré par type de sol
plt.figure(figsize=(11,7))
sns.scatterplot(data=df_pca_sol, x='F1', y='F2', hue='Sol', palette='Set2', alpha=0.6)

#  axes centraux pour repérer l'origine
plt.axhline(0, color='grey', linestyle='--', alpha=0.3)
plt.axvline(0, color='grey', linestyle='--', alpha=0.3)

# Titres
plt.title("Plan factoriel (F1 x F2) par type de sol")
plt.legend(title="Type de sol", bbox_to_anchor=(1.05, 1), loc='upper left')

plt.show()

Même si le nuage de point reste assez illisible, l'ajout des types de sols comme variable explicative permet de faire ressortir des petites informations notamment concernant les sols de type  Urban areas (jaune), Vertisols (vert clair) et Alluvium très groupés en bas, indiquant des profils beaucoup plus homogènes et moins pollués. Les autres types de sols sont parfois légèrement plus pollués (plus d'hétérogénéité) avec des taux plus ou moins variables.

In [ ]:
df_pca_bool = pd.DataFrame({
    'F1': coord[:, 0],
    'F2': coord[:, 1],
    'Historique_Banane': Chlordecone_MTQ['histoBanane_bool']
})

# Génération du nuage de points coloré par l'historique
plt.figure(figsize=(11, 7))

# Nous utilisons 'Historique_Banane' pour 'hue'.
sns.scatterplot(data=df_pca_bool, x='F1', y='F2', hue='Historique_Banane', palette='RdBu', alpha=0.7)

On voit clairement un "bloc" orange (sans banane) en bas à gauche et un "nuage" bleu (avec banane) qui se déplace vers le haut et la droite. Cela prouve que les terrains qui ont reçu de la chlordécone ne ressemblent pas géographiquement aux autres. L'historique de culture est le facteur n°1 qui organise mes données. Les sols ayant accueilli des bananeraies (en bleu) ont des caractéristiques physiques (pente, altitude) très différentes des sols jamais cultivés (en orange). C'est cette différence qui explique la présence ou l'absence de chlordécone aujourd'hui

In [ ]:
for i in range(5):
    Chlordecone_MTQ[f'F{i+1}'] = coord[:, i]

# 2. Groupby sur le COUPLE (Sol + Commune)
df_barycentres = Chlordecone_MTQ.groupby(['Sol_simple', 'COMMU_LAB']).agg({
    'F1': 'mean', 
    'F2': 'mean', 
    'F3': 'mean', 
    'F4': 'mean', 
    'F5': 'mean',
    'Taux_Chlordecone': 'mean'
}).reset_index()

L'objectif de ce bloc de code est de passer d'une analyse individuelle (par parcelle) à une analyse synthétique (par territoire). En statistique, c'est ce qu'on appelle la réduction du "bruit" pour faire émerger un signal clair. Aussi afficher le 30000 points rend le nuage de point illisible. C'est pour ça que nous avons également simplifier ce point en calculant les barycentre de chaque couple commune / type de sol. Ce qui nous permet d'obtenir le graphique ci-dessous : 

In [ ]:
plt.figure(figsize=(12, 12))
sns.scatterplot(data=df_barycentres, x='F3', y='F4', 
                hue='Sol_simple', size='Taux_Chlordecone', 
                sizes=(20, 200), alpha=0.7)

# étiquettes 
for i in df_barycentres.index:
    plt.text(df_barycentres.loc[i, 'F3'], 
             df_barycentres.loc[i, 'F4'], 
             df_barycentres.loc[i, 'COMMU_LAB'],
             fontsize=8)

plt.axhline(0, color='grey', ls='--')
plt.axvline(0, color='grey', ls='--')
plt.title('Barycentres F3 x F4 avec Communes')
plt.show()

Le 3eme et le 4eme axe ne se focalise pas sur le relief ou la chimie mais des éléments assez particuliers et difficilements interprétables mais qui font tout de même ressortir des différences peut être propre aux différentes communes de martiniques et leurs richesse individuels. Néanmoins ce graphique permet de confirmer par rapport à la taille des ronds que les sols de type Vertisol sont les moins pollués au chlordecone.

### V-2. CAH

In [ ]:
colonnes_analyse = quanti_vars + ["histoBanane_bool"]
df_corr = Chlordecone_MTQ[colonnes_analyse]

# Calcul de la matrice
matrix = df_corr.corr()

plt.figure(figsize=(4, 4))
sns.heatmap(matrix, annot=True)
plt.show()

Grâce à la matrice de corrélation, on peut observer une dépendance quasi parfaite entre la pente, la rugosité et tri d'un sol.Ce qui a déjà été soulevé par l'ACP en nous donnant une sorte de caractéristique du relief de certain sol. Mais aussi ue corrélation non négligeable de 0.36 avec la variable histo banane.


In [ ]:
# =================================================================
# CAH SUR BARYCENTRES (SOL + PLUIE) ET SYSTÈME EXPERT
# =================================================================
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib.pyplot as plt
import seaborn as sns

# 1. CALCUL DES BARYCENTRES (Moyenne par type de sol)
df_barycentres = pd.DataFrame(coord[:, :5], columns=['F1','F2','F3','F4','F5'])
df_barycentres['Sol_simple'] = Chlordecone_MTQ['Sol_simple'].values

df_sol_mean = df_barycentres.groupby("Sol_simple")[['F1','F2','F3','F4','F5']].mean()
data_cah = df_sol_mean.values
noms_sols = df_sol_mean.index

# 2. CAH (Méthode de Ward)
Z = linkage(data_cah, method='ward')

# 3. DENDROGRAMME
plt.figure(figsize=(12, 8))
dendrogram(Z, labels=noms_sols, orientation="right", leaf_font_size=11)
plt.title("Classification des types de sols (Profil Pollution & Pluviométrie)")
plt.show()

# 4. CLUSTERING ET RÉINTÉGRATION
df_sol_mean['Cluster_Sol'] = fcluster(Z, t=3, criterion='maxclust')
mapping_sol_risque = df_sol_mean['Cluster_Sol'].to_dict()
Chlordecone_MTQ['Groupe_Risque_Sol'] = Chlordecone_MTQ['Sol_simple'].map(mapping_sol_risque)

# 5. SYSTÈME DE RECOMMANDATIONS AGRONOMIQUES (V3 : SOL + PLUIE + HISTO)
def recommandations_finales_v3(row):
    taux = row['Taux_Chlordecone']
    sol = str(row['Sol_simple']).lower()
    pluie = str(row['RAIN']).lower()
    # AJOUT : On récupère l'historique banane (booléen)
    histo = row['histoBanane_bool'] 
    
    # --- LOGIQUE DE DÉCISION MULTICRITÈRE ---
    
    # CAS 1 : RISQUE MAXIMAL (Historique Banane + Sol à forte rétention)
    if histo == 1 and ("ando" in sol or "niti" in sol):
        return "ALERTE ROUGE : Ancien site bananier sur sol fixateur. Rétention maximale. CULTURES RACINES INTERDITES."

    # CAS 2 : RISQUE ÉLEVÉ (Taux mesuré élevé ou Andosol seul)
    elif taux > 0.1 or "ando" in sol:
        msg = "ZONE CRITIQUE : "
        if "ando" in sol: msg += "Propriétés du sol (Andosol)."
        else: msg += "Taux mesuré > LMR."
        
        if "fort" in pluie:
            return f"{msg} Lessivage important. Préférer cultures hors-sol ou aériennes."
        return f"{msg} Analyse de sol annuelle recommandée."

    # CAS 3 : RISQUE MODÉRÉ (Historique présent mais taux faible ou sol "léger")
    elif histo == 1:
        return "VIGILANCE : Ancien site de traitement. Risque de poches de pollution résiduelles. Tubercules sous surveillance."

    # CAS 4 : ZONE SAINE (Pas d'historique, taux faible)
    else:
        return "ZONE FAVORABLE : Pas d'historique de traitement majeur. Toutes cultures autorisées sous réserve de test."

# Application de la nouvelle logique
Chlordecone_MTQ['PRECONISATIONS'] = Chlordecone_MTQ.apply(recommandations_finales_v3, axis=1)

# 6. VISUALISATION FINALE (Couleur par Risque, Forme par Historique Banane)
plt.figure(figsize=(12, 8))
sns.scatterplot(
    data=Chlordecone_MTQ, 
    x=coord[:, 0], y=coord[:, 1], 
    hue='Groupe_Risque_Sol', 
    style='histoBanane_bool', # La forme (X ou O) indique l'historique banane
    palette='viridis', alpha=0.6, s=100
)

# Affichage des noms des sols sur leurs barycentres
for sol in noms_sols:
    bary = df_sol_mean.loc[sol]
    plt.text(bary['F1'], bary['F2'], sol, fontsize=9, weight='bold', 
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='black'))

plt.title("Plan ACP : Risque Sol (Couleur) et Historique Banane (Forme)")
plt.axhline(0, color='grey', ls='--')
plt.axvline(0, color='grey', ls='--')
plt.legend(title="Risque / Historique (1=Banane)", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

Conclusion Synthétique : De l'Analyse Factorielle à la Décision Agronomique
L'étude croisée de la contamination par la chlordécone en Martinique, à travers l'ACP et la CAH, permet de dégager une structure environnementale cohérente et d'établir un système expert de recommandation.

La position singulière des Andosols confirme leur rôle de réservoir de chlordécone. Leur structure physique favorise une rétention durable du polluant, ce qui en fait des zones de vulnérabilité. L'intégration de la variable RAIN révèle que les pôles de forte pollution coïncident avec les zones de haute pluviométrie. Cette zone du plan factoriel, fortement marquée par la présence d'anciennes bananeraies (symbolisées par les croix sur le graphique), confirme le rôle de ces sols comme réservoirs de chlordécone.

NIVEAU 1 : Risque Faible (Zone Saine)
Profil : Sols qui retiennent peu le produit, zones peu pluvieuses et taux de pollution très bas.

Conseil : Toutes les cultures sont autorisées (tubercules comme l'igname ou la patate douce, fruits et légumes).

🟠 NIVEAU 2 : Risque Modéré (Vigilance)
Profil : Pollution moyenne ou zones très pluvieuses qui favorisent le déplacement du produit vers la plante.

Conseil : Priorité aux cultures aériennes (bananes, arbres fruitiers). Une analyse de sol est obligatoire avant de planter des légumes racines.

🔴 NIVEAU 3 : Risque Critique (Alerte)
Profil : Présence d'Andosols ou taux de pollution dépassant les normes, surtout si la pluie est abondante.

### V-3. Analyses  bivariées

Les politiques de luttes contre la chlordecone dans les antilles ont débuté depuis une dizaine d'année. Un moyen de savoir si les politiques mises en place fonctionnent réellement est de visualisé l'évolution du nombre de parcelles fortement contaminées qui doit normalement baissé au fil du temps. Nous considèrerons une parcelle très contaminée, une parcelle avec un taux supérieures à 1 (source : https://www.geomartinique.fr/accueil/ressources/pollution-des-sols-par-la-chlordecone-en-martinique).

In [ ]:
df_critique = Chlordecone_MTQ[Chlordecone_MTQ['Taux_Chlordecone'] > 1].copy()

# nombre de parcelles par année
evolution_critique = df_critique.groupby('ANNEE').size().reset_index(name='nb_parcelles_critiques')

(
    ggplot(evolution_critique, aes(x='ANNEE', y='nb_parcelles_critiques'))
    + geom_col(fill="#b2182b", alpha=0.8)  
    + geom_text(aes(label='nb_parcelles_critiques'), va='bottom', size=10, nudge_y=0.5)
    + theme_minimal()
    + theme(panel_grid_major_x=element_blank()) 
    + labs(
        title="Nombre de Parcelles fortement contaminées",
        subtitle="Évolution annuelle des détections fortes en Martinique",
        x="Année de prélèvement",
        y="Nombre de parcelles"
    )
    + scale_x_continuous(breaks=evolution_critique['ANNEE'].unique())
)

On constate qu’en 2010, environ 2982 parcelles présentaient des taux relativement élevés de chlordécone. Une baisse importante de ces concentrations est observée en 2014, suivie d’une légère remontée en 2017.
Globalement, cette évolution suggère que les actions mises en place pour réduire la contamination semblent produire des effets positifs. Toutefois, le pic observé en 2017 pourrait s’expliquer par plusieurs facteurs, notamment la maturation de certains sols ou l’évolution des concentrations dans d’autres parcelles initialement moins contaminées.

In [ ]:
# Les 5 sols les plus fréquents
top_sols = Chlordecone_MTQ['Sol_simple'].value_counts().head(5).index
df_sols = Chlordecone_MTQ[Chlordecone_MTQ['Sol_simple'].isin(top_sols)]

plt.figure(figsize=(10, 6))

# 2. Création du graphique (Boxplot)
# showfliers=False permet de cacher les points extrêmes pour mieux voir les boîtes
sns.boxplot(data=df_sols, x='Sol_simple', y='Taux_Chlordecone', palette="Set2")

# 3. Habillage simple
plt.title("Répartition de la pollution par type de sol", fontsize=12)
plt.xlabel("Type de Sol")
plt.ylabel("Taux de Chlordécone")

# Rotation légère pour éviter que les noms se touchent
plt.xticks(rotation=15) 

plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout() # Ajuste automatiquement les marges
plt.show()

## VI- Cartographie

Pour terminer le projet, nous allons proposer une visualisation interactive de la Chlordecone dans les différentes parcelles agricoles des antilles.

In [ ]:
# Lecture du fichier shapefile
# https://fr.moonbooks.org/Articles/Comment-Lire-et-Visualiser-un-Fichier-Shapefile-shp-avec-Python-/#myGallery
shp = gpd.read_file("carto\Martinique.shp")

print(shp.crs)

In [ ]:
print(f"Nombre de lignes (entités) dans le GeoDataFrame : {len(shp)}")

Le nombre de ligne du geodataframe correspond exactement aux nombres de communes de Martinique.

In [ ]:
shp.columns

In [ ]:
shp.plot(figsize=(6, 6), edgecolor='gray', alpha=0.5)
plt.title("Visualisation complète du shapefile")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

In [ ]:
# Calcul du centroïde pour centrer la carte
shp_gps = shp.to_crs(epsg=4326)
center = [shp_gps.geometry.centroid.y.mean(), shp_gps.geometry.centroid.x.mean()]

# Création de la carte
m = folium.Map(location=center,zoom_start=10)

In [ ]:
import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMapWithTime
import unicodedata
import re
from scipy.cluster.hierarchy import linkage, fcluster
from pyproj import Transformer

# =================================================================
# 1. PRÉPARATION ET NETTOYAGE
# =================================================================

def clean_name_complete(nom):
    if not isinstance(nom, str): return ""
    nom = nom.upper().strip()
    nom = ''.join(c for c in unicodedata.normalize('NFD', nom) if unicodedata.category(c) != 'Mn')
    match = re.search(r'[\(\s]+(LE|LA|LES|L|D|DES)[\)]*$', nom)
    if match:
        article = match.group(1)
        reste = nom[:match.start()].strip()
        nom = f"{article} {reste}"
    return nom.replace("-", " ").replace("'", " ").replace("’", " ").strip()

# Conversion coordonnées
transformer = Transformer.from_crs("EPSG:32620", "EPSG:4326", always_xy=True)
lon, lat = transformer.transform(Chlordecone_MTQ["X"].values, Chlordecone_MTQ["Y"].values)
Chlordecone_MTQ["latitude"], Chlordecone_MTQ["longitude"] = lat, lon

Chlordecone_MTQ["COMMUNE_JOIN"] = Chlordecone_MTQ["COMMU_LAB"].apply(clean_name_complete)
shp_gps["COMMUNE_JOIN"] = shp_gps["LIBGEO"].apply(clean_name_complete)

# =================================================================
# 2. CAH ET SYSTÈME EXPERT
# =================================================================

# Calcul des barycentres par sol
df_barycentres = pd.DataFrame(coord[:, :5], columns=['F1','F2','F3','F4','F5'])
df_barycentres['Sol_simple'] = Chlordecone_MTQ['Sol_simple'].values
df_sol_mean = df_barycentres.groupby("Sol_simple").mean()

# CAH
Z = linkage(df_sol_mean.values, method='ward')
df_sol_mean['cluster_id'] = fcluster(Z, t=3, criterion='maxclust')

# Mapping des clusters
dict_clusters = df_sol_mean['cluster_id'].to_dict()
Chlordecone_MTQ['cluster_cah'] = Chlordecone_MTQ['Sol_simple'].map(dict_clusters)

# Ordonner les risques par pollution moyenne pour garantir que Critique = Rouge
ordre_pollu = Chlordecone_MTQ.groupby('cluster_cah')['Taux_Chlordecone'].mean().sort_values().index
risk_names = {ordre_pollu[0]: 'Risque Faible', ordre_pollu[1]: 'Risque Modéré', ordre_pollu[2]: 'Risque Critique'}
Chlordecone_MTQ['Niveau_Risque'] = Chlordecone_MTQ['cluster_cah'].map(risk_names)

# Système Expert : Préconisations agricoles
def recommandations_finales_v5(row):
    taux = row['Taux_Chlordecone']
    sol = str(row['Sol_simple']).lower()
    
    # CRITIQUE : Forte pollution OU Andosols (rétention max)
    if taux > 0.1 or "ando" in sol:
        return "CRITIQUE : Cultures racines INTERDITES."
    # MODÉRÉ
    elif taux > 0.03:
        return "MODÉRÉ : Tubercules sous surveillance."
    # FAIBLE
    else:
        return "FAIBLE : Toutes cultures autorisées."

Chlordecone_MTQ['PRECONISATIONS'] = Chlordecone_MTQ.apply(recommandations_finales_v5, axis=1)

# =================================================================
# 3. FUSION : LE MODE (Pour la couleur de fond)
# =================================================================

def get_mode_risk(x):
    """ Prend le risque majoritaire pour la couleur de fond """
    return x.mode()[0] if not x.mode().empty else "Inconnu"

stats_pollu = Chlordecone_MTQ.groupby("COMMUNE_JOIN").agg({
    "Taux_Chlordecone": "mean",
    "COMMU_LAB": "count",
    "Niveau_Risque": get_mode_risk, # Logique majoritaire pour le fond
    "PRECONISATIONS": lambda x: x.mode()[0] if not x.mode().empty else "N/A"
}).rename(columns={"Taux_Chlordecone": "taux_moyen", "COMMU_LAB": "nb_prelevements"}).reset_index()

# Nettoyage avant fusion
cols_to_drop = ["nb_prelevements", "taux_moyen", "Niveau_Risque", "PRECONISATIONS"]
shp_gps = shp_gps.drop(columns=[c for c in cols_to_drop if c in shp_gps.columns])

# Fusion
shp_gps = shp_gps.merge(stats_pollu, on="COMMUNE_JOIN", how="left").fillna({
    "Niveau_Risque": "Inconnu",
    "taux_moyen": 0,
    "nb_prelevements": 0
})

# =================================================================
# 4. GÉNÉRATION DE LA CARTE (Superposition couches)
# =================================================================

m = folium.Map(location=[14.6415, -61.0242], zoom_start=11, tiles="CartoDB positron")

def get_color(risk):
    return {
        'Risque Critique': '#e74c3c', # Rouge
        'Risque Modéré': '#f39c12',  # Orange
        'Risque Faible': '#27ae60',   # Vert
        'Inconnu': '#bdc3c7'          # Gris
    }.get(risk, 'white')

# --- COUCHE 1 : Les Communes (Fond Majoritaire) ---
folium.GeoJson(
    shp_gps,
    style_function=lambda x: {
        "fillColor": get_color(x['properties']['Niveau_Risque']),
        "color": "black", "weight": 0.5, "fillOpacity": 0.3 # OPACITÉ TRÈS FAIBLE (0.3)
    },
    tooltip=folium.features.GeoJsonTooltip(
        fields=['LIBGEO', 'taux_moyen', 'Niveau_Risque'],
        aliases=['Commune :', 'Moyenne commune :', 'Risque Majoritaire :'],
        localize=True
    )
).add_to(m)

# --- COUCHE 2 : Les Points (Les Petites Taches) ---
# On va créer une couleur spécifique par point basé sur son Taux RÉEL
def get_point_color(taux):
    if taux > 0.1: return '#e74c3c'   # Rouge
    if taux > 0.03: return '#f39c12'  # Orange
    return '#27ae60'                  # Vert

# Sous-ensemble pour accélérer le rendu (si > 10000 points, un échantillon suffit)
points_to_plot = Chlordecone_MTQ.dropna(subset=['latitude', 'longitude'])

for _, row in points_to_plot.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=1.5, # TOUT PETITS CERCLES
        color=get_point_color(row['Taux_Chlordecone']),
        fill=True,
        fill_color=get_point_color(row['Taux_Chlordecone']),
        fill_opacity=0.9,
        weight=0,
        # Infobulle sur le point réel
        tooltip=f"Taux: {row['Taux_Chlordecone']:.4f} mg/kg | Sol: {row['Sol_simple']} | Conseil: {row['PRECONISATIONS']}"
    ).add_to(m)

# Légende HTML adaptée aux deux niveaux
legend_html = '''
     <div style="position: fixed; bottom: 50px; left: 50px; width: 220px; height: 110px; 
     border:2px solid grey; z-index:9999; font-size:12px; background-color:white; padding: 10px; border-radius:5px;">
     <b>Légende Double</b><br>
     <i class="fa fa-square" style="color:#e74c3c; opacity:0.3"></i> Fond Commune : Risque Majoritaire<br>
     <i class="fa fa-circle" style="color:#e74c3c"></i> Point : Pollution réelle mesurée<br>
     <i class="fa fa-square" style="color:#f39c12; opacity:0.3"></i> Fond: Modéré | <i class="fa fa-circle" style="color:#f39c12"></i> Point: Modéré<br>
     <i class="fa fa-square" style="color:#27ae60; opacity:0.3"></i> Fond: Faible | <i class="fa fa-circle" style="color:#27ae60"></i> Point: Faible
     </div>
     '''
m.get_root().html.add_child(folium.Element(legend_html))

folium.LayerControl().add_to(m)
m